# 6. Middleware

This notebook demonstrates how to use middleware or advanced routing and handling.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import SystemMessage, HumanMessage

agent = create_agent(
    model="google_genai:gemini-1.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-1.5-flash-lite",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ],
)

In [5]:
## run with the thread id
config={"configurable":{"thread_id":"test-1"}}

In [8]:
questions = [
    'what is 2+2',
    'what is the capital of France',
    'what is 2 * 2',
    'what is 4*4',
    'while loop in python, how to break the loop when a condition is met',
    'what is the square root of 16'
]

In [10]:

# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is asking a series of basic math and general knowledge questions. The overall task is to answer these questions accurately.\n\n## SUMMARY\n\n- Previously answered basic math and general knowledge questions: "2+2 = 4", "capital of France is Paris", "2 * 2 = 4", and "4 * 4 = 16".\n- In the latest turns, the user asked "what is 2+2", "what is the capital of France", and "what is 2 * 2", which were answered with "2 + 2 = 4", "The capital of France is Paris.", and "2 * 2 = 4" (or equivalent) respectively.\n- No options were rejected; straightforward factual and mathematical questions were answered directly.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- Acknowledge and respond to any new questions or prompts provided by the user.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='56ae6ad2-9e27-4398-a9e7-37d5bc58561f'), AIMessage(content=[{'t

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="google_genai:gemini-1.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-1.5-flash-lite",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# Absolute token limits are supported by Gemini model profiles.
agent = create_agent(
    model="google_genai:gemini-1.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-1.5-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~88 tokens (0.0687%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='c1daedac-a3ea-4d88-bbd0-1b3a0bbd3379'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'SznF22mY': 'EjQKMgERTTIPL8JWnsqkmx26TNggXDTedshYQe/nnfvuIPbpj9s5EZ5SZmbRRY/2bob7dCpp'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8a45-5f1c-7312-8a33-7519e47f4e3e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'SznF22mY', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris: Grand Hotel $350, City Inn $180, Budget Stay $75', name='search_hotels', id='72e6fd4c-6500-4410-b


## Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).
Compliance workflows where human oversight is mandatory.
Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Read an email by ID."""
    return f"Email {email_id}: Subject: Meeting Reminder, Body: Don't forget our meeting at 3 PM."

def send_email_tool(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to} with subject '{subject}' and body '{body}'."



In [32]:
agent=create_agent(
        model="google_genai:gemini-1.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "cancel"]
                },
                "read_email_tool": False
            }
        ),
    ]
)

In [33]:

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [26]:

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d93c45f2-9423-4e26-ac72-bd4418f657cf'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "to": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'KajNUAwk': 'EjQKMgERTTIPWYMbPvpfiSkchc34DzxR96LNQZPkryFT83bvmYtx0ev99Suo9eMOCis2EzNL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8a4b-8092-7771-9e53-a4ea2c34c0c2-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'to': 'john@test.com', 'body': 'How are you?'}, 'id': 'KajNUAwk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 37, 'total_tokens': 174, 'input_token_details': {'cache_read':

In [34]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'Email sent successfully to john@test.com.', 'extras': {'signature': 'EjQKMgERTTIPTWO7kVX2finE/idBy8grOJIbQLcAdpMVFSSLE9Wqlj1RuMxeG4flgiUfnxsk'}}]


## reject


In [36]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-1.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [37]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [38]:
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email was not sent because the tool call was rejected by the user. Let me know if you would like me to try again or help you with something else.', 'extras': {'signature': 'EjQKMgERTTIP4Tsi2l2pha4Nqc4MuIb9l5MD9sFf5H4If0rYtJJQeFhoRBrS5v+foOPkASY2'}}]


In [39]:

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1efc79c2-af2e-42d5-9d91-f1d8ffe9ba2d'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'Czq6rtbr': 'EjQKMgERTTIPy4lD12t+H3Ze9XeBVzG3oTXHFbn04qkWFPK2I+6JxaOuphHPt3MWul6fqF7O'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8a4e-7d80-7533-93a6-a70d781ce85c-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': 'Czq6rtbr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 37, 'total_tokens': 181, 'input_token_details': 

## Edit


In [41]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-1.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [42]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [43]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: [{'type': 'text', 'text': 'I have sent the email to the corrected recipient with a revised subject and body as requested.', 'extras': {'signature': 'EjQKMgERTTIP6wez1ALNQjORmITbEtiBtGo7hI/d61Gl4B+05IrADliuwvfyOcEFeLOPwLeZ'}}]


In [44]:

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='adcf7170-4da1-41a6-804a-f375968ceaed'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "recipient": "wrong@email.com", "subject": "Test"}'}, '__gemini_function_call_thought_signatures__': {'dwdj8n6c': 'EjQKMgERTTIPV8FuMq9HyiQNJzGWoAZXPzg9MIvIu3DanZnwQTObc4w3Lncb11tFeUJ0R2JE'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8a50-0933-7fb3-9af2-3a454b268f0d-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'dwdj8n6c'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 34, 'total_tokens